<a href="https://colab.research.google.com/github/DaviAlves06/ai-pharmacy-agent/blob/main/Agente_Farmac%C3%AAutico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PROJETO PRÁTICO — Agente Farmacêutico Explicador de Bulas (RAG)


Este projeto incorpora:
- Múltiplos documentos
- Chunking mais estratégico
- Enriquecimento com metadados
- Maior controle semântico e rastreabilidade



## Dependências

```bash
!pip install -qU \
  "langchain>=0.2.0,<0.4.0" \
  langchain-community \
  langchain-google-genai \
  chromadb \
  pypdf \
  -qU sentence-transformers
```


In [ ]:
!pip install -qU \
  "langchain>=0.2.0,<0.4.0" \
  langchain-community \
  langchain-google-genai \
  chromadb \
  pypdf \
  -qU sentence-transformers

In [ ]:
# Importa o módulo para manipulação de variáveis de ambiente
import os

# Loader responsável por ler arquivos PDF
from langchain_community.document_loaders import PyPDFLoader

# Responsável por dividir textos grandes em chunks menores
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings com Gemini (Google Generative AI)
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Banco vetorial para armazenamento e busca semântica
from langchain_community.vectorstores import Chroma

# Modelo de linguagem conversacional (Gemini)
from langchain_google_genai import ChatGoogleGenerativeAI

# Cadeia pronta de Perguntas e Respostas com RAG
from langchain.chains import RetrievalQA

In [ ]:
import os
import getpass

# Defina sua chave da Google AI (Gemini) como variável de ambiente
# Gere a chave em: https://ai.google.dev/gemini-api/docs/api-key
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Informe sua GOOGLE_API_KEY: ")


## 1️⃣ Definição do Problema

Bulas farmacêuticas possuem:
- Linguagem técnica
- Grande volume de texto
- Informações sensíveis

O objetivo do agente é **interpretar corretamente as bulas**,
respondendo perguntas **somente com base nos documentos**,
evitando qualquer tipo de alucinação.



## 2️⃣ Seleção e Organização das Bulas




In [ ]:

# Lista com os caminhos dos arquivos PDF das bulas
caminhos_bulas = [
    "dipirona.pdf",
    "paracetamol.pdf"
]

# Lista que armazenará todos os documentos carregados
documentos = []

# Percorre cada bula
for caminho in caminhos_bulas:

    # Cria o loader do PDF
    loader = PyPDFLoader(caminho)

    # Carrega o conteúdo do PDF
    docs = loader.load()

    # Adiciona o nome do medicamento como metadado
    for doc in docs:
        doc.metadata["medicamento"] = caminho.split("/")[-1].replace(".pdf", "")

    # Adiciona os documentos à lista principal
    documentos.extend(docs)

# Exibe a quantidade total de páginas carregadas
len(documentos)


11

In [ ]:
print(documentos)

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'dipirona.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'medicamento': 'dipirona'}, page_content='dipirona monoidratada \n \nBrainfarma Indústria Química e Farmacêutica S.A. \n \n \nComprimido  \n1g'), Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'dipirona.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2', 'medicamento': 'dipirona'}, page_content='dipirona monoidratada – comprimido - Bula para o paciente  1 \n \nI - IDENTIFICAÇÃO DO MEDICAMENTO: \n \ndipirona monoidratada \nMedicamento genérico Lei nº 9.787, de 1999. \n \nAPRESENTAÇÕES \nComprimido. \nEmbalagens contendo 10 ou 100 comprimidos.  \n \nVIA DE ADMINISTRAÇÃO: ORAL \n \nUSO ADULTO E PEDIÁTRICO ACIMA DE 15 ANOS. \n \nCOMPOSIÇÃO \nCada comprimido contém: \ndipirona monoidratada ....................................................................


## 3️⃣ Extração, Limpeza e Chunking



In [ ]:

# Cria o objeto responsável pelo chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # Tamanho máximo de cada chunk
    chunk_overlap=150   # Sobreposição entre chunks
)

# Divide os documentos em chunks
chunks = text_splitter.split_documents(documentos)

# Quantidade total de chunks gerados
len(chunks)


61

In [ ]:
chunks

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'dipirona.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'medicamento': 'dipirona'}, page_content='dipirona monoidratada \n \nBrainfarma Indústria Química e Farmacêutica S.A. \n \n \nComprimido  \n1g'),
 Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'dipirona.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2', 'medicamento': 'dipirona'}, page_content='dipirona monoidratada – comprimido - Bula para o paciente  1 \n \nI - IDENTIFICAÇÃO DO MEDICAMENTO: \n \ndipirona monoidratada \nMedicamento genérico Lei nº 9.787, de 1999. \n \nAPRESENTAÇÕES \nComprimido. \nEmbalagens contendo 10 ou 100 comprimidos.  \n \nVIA DE ADMINISTRAÇÃO: ORAL \n \nUSO ADULTO E PEDIÁTRICO ACIMA DE 15 ANOS. \n \nCOMPOSIÇÃO \nCada comprimido contém: \ndipirona monoidratada ...................................................................


## 4️⃣ Enriquecimento com Metadados


In [ ]:

# Percorre cada chunk para classificar semanticamente seu conteúdo
for chunk in chunks:

    # Normaliza o texto para facilitar as verificações
    texto = chunk.page_content.lower()

    # Identificação do medicamento
    if "identificação do medicamento" in texto or "composição" in texto:
        chunk.metadata["categoria"] = "identificacao"

    # Indicações terapêuticas
    elif "indicação" in texto or "para que este medicamento é indicado" in texto:
        chunk.metadata["categoria"] = "indicacao"

    # Funcionamento do medicamento
    elif "como este medicamento funciona" in texto or "ação" in texto:
        chunk.metadata["categoria"] = "como_funciona"

    # Contraindicações
    elif "contraindicação" in texto or "quando não devo usar" in texto:
        chunk.metadata["categoria"] = "contraindicacao"

    # Advertências e precauções
    elif "advertência" in texto or "precaução" in texto or "o que devo saber antes de usar" in texto:
        chunk.metadata["categoria"] = "advertencias_precaucoes"

    # Interações medicamentosas
    elif "interação" in texto or "interações medicamentosas" in texto:
        chunk.metadata["categoria"] = "interacoes"

    # Posologia e modo de uso
    elif "dose" in texto or "posologia" in texto or "como devo usar" in texto:
        chunk.metadata["categoria"] = "posologia_modo_uso"

    # Reações adversas
    elif "reações adversas" in texto or "quais os males" in texto:
        chunk.metadata["categoria"] = "reacoes_adversas"

    # Armazenamento
    elif "onde, como e por quanto tempo posso guardar" in texto or "armazenar" in texto:
        chunk.metadata["categoria"] = "armazenamento"

    # Superdosagem
    elif "quantidade maior do que a indicada" in texto or "superdosagem" in texto:
        chunk.metadata["categoria"] = "superdosagem"

    # Conteúdo geral / administrativo
    else:
        chunk.metadata["categoria"] = "geral"



In [ ]:
import random

# Seleciona dois chunks aleatórios
chunks_aleatorios = random.sample(chunks, 2)

# Imprime os metadados e um trecho do conteúdo
for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(f"\n--- Chunk Aleatório {i} ---")
    print("Metadados:")
    print(chunk.metadata)
    print("\nConteúdo (início):")
    print(chunk.page_content[:300])



--- Chunk Aleatório 1 ---
Metadados:
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'dipirona.pdf', 'total_pages': 10, 'page': 2, 'page_label': '3', 'medicamento': 'dipirona', 'categoria': 'como_funciona'}

Conteúdo (início):
aproximadamente 4 horas. 
 
3. QUANDO NÃO DEVO USAR ESTE MEDICAMENTO? 
A dipirona monoidratada não deve ser utilizada caso você tenha: 
- alergia ou intolerância à dipirona ou a qualquer um dos componentes da formulação ou a outras 
pirazolonas ou a pirazolidinas (ex.: fenazona, propifenaz ona, isop

--- Chunk Aleatório 2 ---
Metadados:
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2017-12-21T09:26:33-02:00', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'moddate': '2017-12-21T09:26:33-02:00', 'title': 'PARACETAMOL.indd', 'trapped': '/False', 'source': 'paracetamol.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'medicamento': 'pa


## 5️⃣ Geração de Embeddings e Banco Vetorial


In [ ]:
#!pip install chromadb


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embeddings locais (HuggingFace) – estáveis, gratuitos e sem depender de API externa
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Cria o banco vetorial com os chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_bulas"
)

/tmp/ipython-input-3405981122.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



## 6️⃣ Recuperação de Contexto (Retriever)


In [ ]:

# Cria o retriever para busca semântica
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}  # Número de chunks retornados
)



## 7️⃣ Integração com Pipeline RAG


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,   # respostas mais factuais
    transport="rest",  # evita gRPC e o erro 'Illegal metadata'
    timeout=60,        # tempo máx. de cada chamada (segundos)
    max_retries=1,     # no máx. 1 retry rápido; evita ficar 10+ minutos
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)


## 8️⃣ Testes e Validação das Respostas


In [ ]:
# Pergunta de teste
pergunta = "Quais são as contraindicações da dipirona?"

# Executa a pergunta no agente RAG (forma atual)
resposta = qa_chain.invoke(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

# Percorre os documentos recuperados
for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")

    # Conteúdo recuperado
    print("\nConteúdo do chunk:")
    print(doc.page_content)
    print("\n")


Pergunta:
Quais são as contraindicações da dipirona?

Resposta do Agente:
Com base nas informações fornecidas, as contraindicações da dipirona são:

*   **Amamentação:** Deve ser evitada durante e por até 48 horas após o uso da dipirona monoidratada.
*   **Crianças:**
    *   Menores de 3 meses ou pesando menos de 5kg não devem ser tratadas com dipirona.
    *   A dipirona monoidratada em comprimido não é recomendada para menores de 15 anos.

Trechos utilizados como contexto:

--- Trecho 1 ---
Medicamento: dipirona
Categoria: como_funciona
Documento: dipirona.pdf
Página: 3

Conteúdo do chunk:
A amamentação deve ser evitada durante e por até 48 horas após o uso d a dipirona monoidratada . A 
dipirona é eliminada no leite materno. 
 
Pacientes idosos:  deve-se considerar a possibilidade da s funções do fígado e dos rins estarem 
prejudicadas. 
Crianças: menores de 3 meses ou pesando menos de 5kg não devem ser tratadas com dipirona. 
A dipirona  monoidratada comprimido não é recomendada p

In [ ]:
# Pergunta de teste 1
pergunta = "Quais são as recomendações do paracetamol?"

# Executa a pergunta no agente RAG
resposta = qa_chain.invoke(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Medicamento: {doc.metadata.get('medicamento', 'N/A')}")
    print(f"Categoria: {doc.metadata.get('categoria', 'N/A')}")
    print(f"Documento: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("\nConteúdo do chunk:")
    print(doc.page_content)
    print("\n")


Pergunta:
Quais são as recomendações do paracetamol?

Resposta do Agente:
O texto fornecido lista as interações e contraindicações do paracetamol, mas não apresenta recomendações gerais de uso (como dosagem, frequência ou indicações).

As informações disponíveis são:

**INTERAÇÕES:**
*   Doses elevadas potencializam a ação dos anticoagulantes cumarínicos e indandiônicos.
*   Aumenta a meia-vida do cloranfenicol de 3,25 para 15 horas.
*   Altera os níveis plasmáticos do diflunisal.
*   Aumenta os riscos dos salicilatos.
*   Álcool e anticonvulsivantes realçam seus efeitos tóxicos.
*   Sua depuração metabólica é acelerada em mulheres que tomam anticoncepcionais orais.

**CONTRA-INDICAÇÕES:**
*   Portadores de hepatopatia (doença no fígado).

Trechos utilizados como contexto:

--- Trecho 1 ---
Medicamento: dipirona
Categoria: como_funciona
Documento: dipirona.pdf
Página: 4

Conteúdo do chunk:
medicação. 
Não há estudos dos efeitos d a dipirona monoidratada comprimido administrada por vias